# Black-box transfer role tables, matrix, and scatter

Notebook driver for black-box transfer analysis. Edit the configuration cell only. The helper code lives in `analytics.attacks.transfer_role_tools`.

It can generate:

- **Top damaging sources by target**: `target | top 1 source (metric) | top 2 source | top 3 source`
- **Top robust targets by source**: `source | top 1 robust target (metric) | top 2 robust target | top 3 robust target`
- **Attack matrix** at family / architecture / variant grain
- **Source-vs-target scatter profile** at family / architecture / variant grain


In [1]:
import sys
from pathlib import Path

# Add src/ to sys.path before importing analytics.
_project_root = Path.cwd().resolve()
while _project_root != _project_root.parent and not (_project_root / "src" / "analytics").is_dir():
    _project_root = _project_root.parent
if not (_project_root / "src" / "analytics").is_dir():
    raise FileNotFoundError(
        "Could not find the project root (expected src/analytics/). "
        "Set the notebook working directory to the repository root or a subdirectory."
    )

src_path = _project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print(f"Project root: {_project_root}")


Project root: C:\Users\fwilh\Desktop\QB-PIER


In [ ]:
import pandas as pd

from analytics.paths import attack_experiment_dir, output_dir
from analytics.tables.model_blacklist import MODEL_BLACKLIST
from analytics.attacks.transfer_role_tools import (
    TransferSelection,
    load_or_build_pair_metrics,
    build_top_sources_by_target,
    build_top_robust_targets_by_source,
    build_attack_matrix,
    build_source_target_scatter,
    plot_attack_matrix,
    plot_source_target_scatter,
    write_tables,
)


## Configuration

Edit this cell.

### Grain

- `family`: rows/items are `Classical`, `Hybrid`, `Quantum`
- `architecture`: rows/items are `Classical/CNN`, `Hybrid/HVQL`, etc.
- `variant`: rows/items are full model identifiers

### Comparison mode

- `inter_variant`: compare all variants with each other, excluding exact self-attacks only
- `inter_architecture`: exclude source/target pairs from the same architecture
- `inter_model`: exclude all source/target pairs from the same family. This is equivalent to inter-family transfer in this notebook convention.

### Metric

Typical choices:

- `Transfer ASR` — higher means more damaging
- `Transfer Robust Accuracy` — lower means more damaging
- `Accuracy Drop` — higher means more damaging
- `Mean Confidence Drop`, `Mean Margin Drop` — diagnostic score movement


In [ ]:
# Main display grain. Usually keep source and target grain equal.
GRAIN = "architecture"          # "family", "architecture", or "variant"
SOURCE_GRAIN = GRAIN
TARGET_GRAIN = GRAIN

COMPARISON_MODE = "inter_architecture"  # "inter_variant", "inter_architecture", "inter_model"
METRIC = "Transfer ASR"
TOP_K = 3

# Optional focused filters. Leave as None for full zoo.
SOURCE_FAMILY_FILTER = None       # e.g. "Classical", "Hybrid", "Quantum"
TARGET_FAMILY_FILTER = None
SOURCE_ARCHITECTURE_FILTER = None # e.g. "CNN", "VQM", "HVQL"
TARGET_ARCHITECTURE_FILTER = None
SOURCE_CONTAINS = None            # substring in full source model name
TARGET_CONTAINS = None

# Optional blacklist extension in addition to MODEL_BLACKLIST.
EXTRA_MODEL_BLACKLIST = []

# Pair metrics cache. Set to a CSV/parquet path if you already built pair metrics.
PAIR_METRICS_CACHE = None
REBUILD_PAIR_METRICS = False

WRITE_OUTPUTS = True
PLOT_MATRIX = True
PLOT_SCATTER = True

OUTPUT_NAME = f"transfer_roles_{GRAIN}_{COMPARISON_MODE}_{METRIC.replace(' ', '_')}"
OUT_DIR = output_dir(_project_root, "attack_transfer_role_tables") / OUTPUT_NAME
OUT_DIR


In [ ]:
pair_df = load_or_build_pair_metrics(
    pair_metrics_cache=Path(PAIR_METRICS_CACHE) if PAIR_METRICS_CACHE else None,
    rebuild=REBUILD_PAIR_METRICS,
    attack_dir=attack_experiment_dir(_project_root),
    pair_metrics_output=OUT_DIR / "pair_metrics.csv",
    model_blacklist=set(MODEL_BLACKLIST) | set(EXTRA_MODEL_BLACKLIST),
)

print(pair_df.shape)
pair_df.head()


In [ ]:
selection = TransferSelection(
    source_grain=SOURCE_GRAIN,
    target_grain=TARGET_GRAIN,
    comparison_mode=COMPARISON_MODE,
    metric=METRIC,
    top_k=TOP_K,
    model_blacklist=frozenset(set(MODEL_BLACKLIST) | set(EXTRA_MODEL_BLACKLIST)),
    source_family_filter=SOURCE_FAMILY_FILTER,
    target_family_filter=TARGET_FAMILY_FILTER,
    source_architecture_filter=SOURCE_ARCHITECTURE_FILTER,
    target_architecture_filter=TARGET_ARCHITECTURE_FILTER,
    source_contains=SOURCE_CONTAINS,
    target_contains=TARGET_CONTAINS,
)
selection


In [ ]:
top_sources_wide, top_sources_long = build_top_sources_by_target(pair_df, selection, rank_mode="damage")
robust_targets_wide, robust_targets_long = build_top_robust_targets_by_source(pair_df, selection)
attack_matrix = build_attack_matrix(pair_df, selection)
scatter_profile = build_source_target_scatter(pair_df, selection)

print("Top damaging sources by target")
display(top_sources_wide)

print("Top robust targets by source")
display(robust_targets_wide)

print("Attack matrix")
display(attack_matrix)

print("Source-target scatter profile")
display(scatter_profile)


In [ ]:
if WRITE_OUTPUTS:
    paths = write_tables(
        OUT_DIR,
        top_sources_wide=top_sources_wide,
        top_sources_long=top_sources_long,
        robust_targets_wide=robust_targets_wide,
        robust_targets_long=robust_targets_long,
        matrix=attack_matrix,
        scatter_profile=scatter_profile,
        prefix=OUTPUT_NAME,
    )
    for name, path in paths.items():
        print(f"{name}: {path}")


In [ ]:
if PLOT_MATRIX:
    fig, ax = plot_attack_matrix(
        attack_matrix,
        title=f"Attack matrix — {TARGET_GRAIN}/{SOURCE_GRAIN}, {COMPARISON_MODE}, {METRIC}",
    )
    if WRITE_OUTPUTS:
        fig.savefig(OUT_DIR / f"{OUTPUT_NAME}_attack_matrix.png", dpi=220, bbox_inches="tight")

if PLOT_SCATTER:
    fig, ax = plot_source_target_scatter(
        scatter_profile,
        title=f"Source vs target transfer roles — {GRAIN}, {COMPARISON_MODE}, {METRIC}",
        annotate=(GRAIN != "variant"),
    )
    if WRITE_OUTPUTS:
        fig.savefig(OUT_DIR / f"{OUTPUT_NAME}_source_target_scatter.png", dpi=220, bbox_inches="tight")


## Quick recipes

### Top sources per target architecture, excluding same architecture

```python
GRAIN = "architecture"
COMPARISON_MODE = "inter_architecture"
METRIC = "Transfer ASR"
```

### Top robust targets per source family, excluding same family

```python
GRAIN = "family"
COMPARISON_MODE = "inter_model"
METRIC = "Transfer ASR"
```

### Variant-level table, allowing cross-variant attacks inside same architecture

```python
GRAIN = "variant"
COMPARISON_MODE = "inter_variant"
METRIC = "Transfer ASR"
```

### Robust-accuracy version instead of ASR

```python
METRIC = "Transfer Robust Accuracy"
```

For `Transfer Robust Accuracy`, the most damaging sources are ranked by **lowest** robust accuracy, while robust targets are ranked by **highest** robust accuracy.
